# Mexico City (Iztapalapa) Subsidence — Complete Rebuild, Search to LOS

**Full rebuild after real data loss** — nothing assumed to exist on disk.
Every step from search through the final displacement map, every real fix
from this whole project wired in: `select_consistent_geometry()` for real
track filtering, `extract_consistent_stack()` for real sub-swath
consistency, the `row_offset` deburst alignment fix, real per-burst-overlap
ESD, the fixed (fast, correct) topographic-phase and atmospheric
regression, real orbit-based coregistration.

**Why this AOI, as a real positive control**: urban, stable scatterers,
flat lakebed basin — one of ESA's own two official SNAP-StaMPS validation
sites (Foumelis et al. 2018), with a real, published subsidence signal to
check against (Cigna & Tapete 2021, Remote Sensing of Environment: peak
-39.1 cm/year in Iztapalapa, from 300+ scenes, 2014-2020).

In [1]:
import numpy as np
import rasterio
from pathlib import Path
from datetime import datetime
from itertools import combinations

from pygeofetch import PyGeoFetch
from pygeofetch.models import BoundingBox
from pygeofetch.models.search_query import SearchQuery
from pygeofetch.models.download_task import DownloadOptions
from pygeofetch.processing.preprocessor import Preprocessor
from pygeofetch.core.orbits import fetch_orbit_file
from pygeofetch.insar import (
    SLCExtractor, InterferogramGenerator, SBASTimeSeries,
    AtmosphericCorrector, select_consistent_geometry,
)
from pygeofetch.insar.timeseries import InterferogramPair
from pygeofetch.insar.geolocation import (
    parse_orbit_file, perpendicular_baseline, los_to_vertical_displacement,
    geodetic_to_ecef, find_zero_doppler_time, interpolate_orbit_state,
)
from pygeofetch.insar.unwrap import PhaseUnwrapper, multilook, bridge_unwrap_regions
from pygeofetch.insar.validate import DataValidator
from pygeofetch.viz.map import MapViewer

client = PyGeoFetch()
output_dir = Path("data/mexico_city_insar")
output_dir.mkdir(parents=True, exist_ok=True)

WAVELENGTH_M = 0.05546576
INCIDENCE_ANGLE_DEG = 39.0
CERRO_LAT, CERRO_LON = 19.34384, -99.09046  # real, documented stable ground

aoi_bbox = BoundingBox(min_lon=-99.183-0.037, max_lon=-99.003-0.037, min_lat=19.278, max_lat=19.438)

16:08:05 INFO [      engine] PyGeoFetch ready


## 1. Search — wide window, then real track filtering

`select_consistent_geometry()` groups by real track and keeps only the
largest same-track group.

In [2]:
search_query = SearchQuery(
    bbox=aoi_bbox, start_date="2024-10-01", end_date="2025-01-31",
    product_type="SLC", max_results=50,satellites=["Sentinel-1A"], platform="Sentinel-1"
)
search_results = client.search(search_query, providers=["copernicus"])
print(f"Real search: {len(search_results)} Sentinel-1 SLC scenes found")

by_date = {}
for r in search_results:
    label = str(r.datetime)[:10]
    if label not in by_date:
        by_date[label] = r

selected, geometry_report = select_consistent_geometry(list(by_date.values()),max_scenes=6,preferred_track=41)
print(f"\nReal track kept: {geometry_report['track']}")
print(f"Real satellites present: {geometry_report['satellites']}")
print(f"Same-geometry scenes ({len(selected)}): {[str(s.datetime)[:10] for s in selected]}")
if geometry_report["dropped"]:
    print(f"Dropped (different real track): {geometry_report['dropped']}")

┌ SEARCH PARAMETERS ───────────────────────────────────────────────────────┐
│ Providers  : copernicus                                                  │
│ BBox       : [-99.220, 19.278, -99.040, 19.438]                          │
│ Date range : 2024-10-01  →  2025-01-31                                   │
│ Cloud max  : 100%                                                        │
│ Product    : SLC                                                         │
└──────────────────────────────────────────────────────────────────────────┘
16:08:11 INFO [  copernicus] Authenticated with Copernicus Data Space as 'appiahkubis14@gmail.com'
16:08:15 INFO [  copernicus] Real unit-level satellite filter (['S1A']): 38/38 results kept.
  ✓  copernicus                      38 scenes   5.3s
┌────────────────────────────────────────────┬────────────┬────────────────┬────────┬─────────┬──────────────┬─────────────┬───────┬───────┬──────────────────────┐
│                  SCENE ID                  │    D

In [3]:
import json
import tempfile

aoi_geojson = {
    "type": "FeatureCollection",
    "features": [{
        "type": "Feature",
        "properties": {"name": "AOI"},
        "geometry": {
            "type": "Polygon",
            "coordinates": [[
                [aoi_bbox.min_lon, aoi_bbox.min_lat],
                [aoi_bbox.max_lon, aoi_bbox.min_lat],
                [aoi_bbox.max_lon, aoi_bbox.max_lat],
                [aoi_bbox.min_lon, aoi_bbox.max_lat],
                [aoi_bbox.min_lon, aoi_bbox.min_lat],
            ]],
        },
    }],
}
aoi_path = Path(tempfile.mkdtemp()) / "aoi.geojson"
aoi_path.write_text(json.dumps(aoi_geojson))

mv = MapViewer(center=((aoi_bbox.min_lat + aoi_bbox.max_lat) / 2, (aoi_bbox.min_lon + aoi_bbox.max_lon) / 2), zoom=10)
mv.add_basemap("SATELLITE")
mv.add_search_results(selected)
mv.add_vector(str(aoi_path), layer_name="AOI", style={"color": "yellow", "fillOpacity": 0, "weight": 3})
mv.show()

16:08:21 INFO [         map] Vector layer added: search_results
16:08:21 INFO [         map] Vector layer added: AOI


Map(center=[19.357999999999997, -99.13000000000001], controls=(ZoomControl(options=['position', 'zoom_in_text'…

## 2. Download the real, filtered scenes

In [ ]:
raw_dir = output_dir / "raw"
download_results_list = client.download(selected, destination=raw_dir, options=DownloadOptions(parallel=3, resume=True))
download_results = {str(s.datetime)[:10]: dr for s, dr in zip(selected, download_results_list)}
extracted_dates = list(download_results.keys())
print(f"Real downloads complete: {len(download_results)} scenes: {extracted_dates}")

⬇ 6 scenes  →  data/mexico_city_insar/raw              0/6  [00:00]

1bf924f2-6a2c-40c2-a67b-1d83aabec671: 0.00B [00:00, ?B/s]

bf09783d-15ed-4818-9e16-ec33ea53bbbc: 0.00B [00:00, ?B/s]

b98c1f73-21f9-40e4-8562-eadaf6294910: 0.00B [00:00, ?B/s]

16:22:39 WARN [  copernicus] Download attempt 1/3 failed for bf09783d-15ed-4818-9e16-ec33ea53bbbc: peer closed connection without sending complete message body (received 47677295 bytes, expected 7522679166). Resuming from 32MB in 15s...
16:25:35 WARN [  copernicus] Download attempt 1/3 failed for 1bf924f2-6a2c-40c2-a67b-1d83aabec671: peer closed connection without sending complete message body (received 42558320 bytes, expected 7522692914). Resuming from 32MB in 15s...
16:27:06 WARN [  copernicus] Download attempt 2/3 failed for bf09783d-15ed-4818-9e16-ec33ea53bbbc: peer closed connection without sending complete message body (received 4236068 bytes, expected 7522679166). Resuming from 0MB in 30s...
16:29:59 WARN [  copernicus] Download attempt 2/3 failed for 1bf924f2-6a2c-40c2-a67b-1d83aabec671: peer closed connection without sending complete message body (received 4095168 bytes, expected 7522692914). Resuming from 0MB in 30s...
16:30:04 WARN [  copernicus] Download attempt 1/3 failed

ac860b13-9d0e-4edc-be10-b73da97066f8: 0.00B [00:00, ?B/s]

16:38:28 INFO [  copernicus] Authenticated with Copernicus Data Space as 'appiahkubis14@gmail.com'
16:38:29 WARN [  downloader] Download attempt 1/4 for 'b98c1f73-21f9-40e4-8562-eadaf6294910' failed validation: 401 Unauthorized — token refreshed, retrying. Retrying in 0.7s...
16:41:33 WARN [  copernicus] Download attempt 1/3 failed for bf09783d-15ed-4818-9e16-ec33ea53bbbc: peer closed connection without sending complete message body (received 15486243 bytes, expected 7522679166). Resuming from 0MB in 15s...
16:43:35 WARN [  copernicus] Download attempt 1/3 failed for ac860b13-9d0e-4edc-be10-b73da97066f8: peer closed connection without sending complete message body (received 8535637 bytes, expected 7522569871). Resuming from 0MB in 15s...
16:43:37 WARN [  copernicus] Download attempt 1/3 failed for b98c1f73-21f9-40e4-8562-eadaf6294910: peer closed connection without sending complete message body (received 16588012 bytes, expected 7522571945). Resuming from 0MB in 15s...
16:46:07 WARN [ 

## 3. Real orbit files

In [ ]:
orbit_dir = output_dir / "orbits"
orbit_dir.mkdir(parents=True, exist_ok=True)

orbit_files = {}
for label, dr in download_results.items():
    try:
        orbit_files[label] = fetch_orbit_file(
            product_name=Path(dr.output_path).name, output_dir=str(orbit_dir), orbit_type="precise",
        )
        print(f"  {label}: {Path(orbit_files[label]).name}")
    except Exception as exc:
        print(f"  {label}: orbit download failed -- {exc}")

print(f"\n{len(orbit_files)}/{len(download_results)} real orbit files ready")

## 4. Real DEM (OpenTopography)

In [ ]:
dem_dir = output_dir / "dem"
dem_dir.mkdir(parents=True, exist_ok=True)

dem_results = client.search(SearchQuery(bbox=aoi_bbox, product_type="DEM"), providers=["opentopography"])
if not dem_results:
    raise RuntimeError("No real DEM found for this AOI -- check opentopography credentials/coverage")

raw_dem_path = client.download(dem_results[:1], destination=dem_dir)[0].output_path
dem_path = Preprocessor().clip(raw_dem_path, bbox=aoi_bbox, output=str(dem_dir / "dem_clipped.tif")).output_path
print(f"Real DEM ready: {dem_path}")

## 5. Real extraction — sub-swath forced consistent, unreliable crops auto-rejected

In [ ]:
extractor = SLCExtractor(polarisation="VV")
scenes = {label: download_results[label].output_path for label in extracted_dates}

extracted_slcs, extraction_report = extractor.extract_consistent_stack(scenes, aoi_bbox, output_dir / "slc")

print(f"Reference: {extraction_report['reference']}, matched real sub-swath: {extraction_report['matched_swath']} "
      f"({extraction_report['reference_rows']} real rows)")
if extraction_report["excluded"]:
    print(f"Excluded: {extraction_report['excluded']}")

orbit_files = {k: v for k, v in orbit_files.items() if k in extracted_slcs}
download_results = {k: v for k, v in download_results.items() if k in extracted_slcs}
extracted_dates = list(extracted_slcs.keys())
# Real, full acquisition timestamps -- built once from `selected` (real
# SatelliteData results), not re-searched per label.
all_acquisition_times = {str(s.datetime)[:10]: str(s.datetime) for s in selected}
acquisition_times = {label: all_acquisition_times[label] for label in extracted_dates}
print(f"\n{len(extracted_slcs)} real, reliable scenes kept: {extracted_dates}")

## 6. Interferogram formation — every real pair, complete verified pipeline

In [ ]:
ifg_gen = InterferogramGenerator(
    coherence_window=5, esd_enabled=True, use_gpu=False,
    use_real_burst_processing=True, remove_flat_earth_phase=True,
)
LOOKS_AZ, LOOKS_RG = 2, 1

interferograms = {}
for d1, d2 in combinations(extracted_dates, 2):
    coreg_kwargs = {}
    if all(d in download_results and d in orbit_files for d in (d1, d2)):
        coreg_kwargs = dict(
            reference_safe_zip=download_results[d1].output_path, secondary_safe_zip=download_results[d2].output_path,
            reference_orbit_file=orbit_files[d1], secondary_orbit_file=orbit_files[d2],
        )
    try:
        result = ifg_gen.process_pair(
            reference=extracted_slcs[d1], secondary=extracted_slcs[d2], dem=dem_path,
            reference_date=d1, secondary_date=d2, looks_azimuth=LOOKS_AZ, looks_range=LOOKS_RG,
            apply_goldstein_filter=True, goldstein_alpha=0.6, **coreg_kwargs,
        )
    except ValueError as exc:
        print(f"  {d1} -> {d2}: REJECTED -- {exc}")
        continue
    days = abs((datetime.fromisoformat(d2).date() - datetime.fromisoformat(d1).date()).days)
    interferograms[(d1, d2)] = result
    result.save(output_dir / "interferograms" / f"{d1}_{d2}", auto_visualize=True)
    print(f"  {d1} -> {d2} ({days:3d}d): coherence={result.coherence.mean():.3f}")

print(f"\n{len(interferograms)} real pairs formed")

## 7. Atmospheric correction

Elevation-correlated, real fixed circular-frequency-search regression --
no external credentials needed.

In [ ]:
atm_corrector = AtmosphericCorrector(method="elevation")
corrected_interferograms = {}

for (d1, d2), result in interferograms.items():
    phase = np.angle(result.interferogram)
    corrected, meta = atm_corrector.correct(phase, dem=dem_path, return_metadata=True)
    corrected_interferograms[(d1, d2)] = corrected
    print(f"  {d1} -> {d2}: correction_applied={meta.get('correction_applied')}, R\u00b2={meta.get('r_squared')}")

## 8. Phase unwrapping — every real pair

In [ ]:
unwrapper = PhaseUnwrapper(cost_mode="defo", init_method="mcf")
UNWRAP_LOOKS_AZ, UNWRAP_LOOKS_RG = 8, 4
TOTAL_LOOKS = LOOKS_AZ * LOOKS_RG * UNWRAP_LOOKS_AZ * UNWRAP_LOOKS_RG

unwrapped_results, conncomp_results, reliability = {}, {}, {}

for (d1, d2) in interferograms:
    phase = corrected_interferograms[(d1, d2)]
    coherence = interferograms[(d1, d2)].coherence
    phase_ml = multilook(phase, UNWRAP_LOOKS_AZ, UNWRAP_LOOKS_RG, wrapped_phase=True)
    coh_ml = multilook(coherence, UNWRAP_LOOKS_AZ, UNWRAP_LOOKS_RG, wrapped_phase=False)
    unwrapped, conncomp = unwrapper.unwrap(
        phase_ml, coh_ml, nlooks=float(TOTAL_LOOKS), min_conncomp_frac=0.001, min_region_size=100,
    )
    unwrapped_results[(d1, d2)] = unwrapped
    conncomp_results[(d1, d2)] = conncomp
    reliability[(d1, d2)] = 100 * np.mean(conncomp > 0)
    print(f"  {d1} -> {d2}: coherence={coh_ml.mean():.3f}, reliable={reliability[(d1,d2)]:5.1f}%")

print(f"\nMean reliable coverage: {np.mean(list(reliability.values())):.1f}%")

## 9. Baseline-optimized network

Real, package-level `perpendicular_baseline()`.

In [ ]:
ground_point = geodetic_to_ecef(CERRO_LAT, CERRO_LON, 0.0)

baselines = []
for d1, d2 in interferograms:
    if d1 not in orbit_files or d2 not in orbit_files:
        continue
    ref_orbit = parse_orbit_file(orbit_files[d1])
    sec_orbit = parse_orbit_file(orbit_files[d2])
    t_ref = find_zero_doppler_time(*ref_orbit, ground_point, datetime.fromisoformat(acquisition_times[d1]))
    t_sec = find_zero_doppler_time(*sec_orbit, ground_point, datetime.fromisoformat(acquisition_times[d2]))
    pos_ref, _ = interpolate_orbit_state(*ref_orbit, t_ref)
    pos_sec, _ = interpolate_orbit_state(*sec_orbit, t_sec)
    b_perp = perpendicular_baseline(pos_ref, pos_sec, ground_point)
    baselines.append((d1, d2, b_perp))
    print(f"  {d1} -> {d2}: baseline={b_perp:.1f}m")

baselines_sorted = sorted(baselines, key=lambda x: x[2])
parent = {d: d for d in extracted_dates}
def find(d):
    while parent[d] != d: d = parent[d]
    return d

network_pairs = []
for d1, d2, b in baselines_sorted:
    r1, r2 = find(d1), find(d2)
    if r1 != r2:
        parent[r1] = r2
        network_pairs.append((d1, d2))

connected = {d for pair in network_pairs for d in pair}
print(f"\nReal baseline-optimized network: {len(network_pairs)} pairs, {len(connected)}/{len(extracted_dates)} dates connected")

## 10. Real, georeferenced reference pixel — Cerro de la Estrella

In [ ]:
reference_pair = next(iter(interferograms))
reference_transform = interferograms[reference_pair].profile["transform"]

cerro_row_native, cerro_col_native = rasterio.transform.rowcol(reference_transform, CERRO_LON, CERRO_LAT)
ref_row_ml = cerro_row_native // (LOOKS_AZ * UNWRAP_LOOKS_AZ)
ref_col_ml = cerro_col_native // (LOOKS_RG * UNWRAP_LOOKS_RG)

min_r = min(u.shape[0] for k, u in unwrapped_results.items() if k in network_pairs) if network_pairs else min(u.shape[0] for u in unwrapped_results.values())
min_c = min(u.shape[1] for k, u in unwrapped_results.items() if k in network_pairs) if network_pairs else min(u.shape[1] for u in unwrapped_results.values())
REF_PIXEL = (min(max(ref_row_ml, 0), min_r - 1), min(max(ref_col_ml, 0), min_c - 1))
print(f"Real, georeferenced reference pixel: {REF_PIXEL}")

## 11. Bridging — exclude unreliable pairs, never corrupt the rest

In [ ]:
sbas_pairs, excluded_pairs = [], []

for (d1, d2) in network_pairs:
    unwrapped = unwrapped_results[(d1, d2)]
    conncomp = conncomp_results[(d1, d2)]
    coherence_ml = multilook(interferograms[(d1, d2)].coherence, UNWRAP_LOOKS_AZ, UNWRAP_LOOKS_RG, wrapped_phase=False)

    unwrapped_c = unwrapped[:min_r, :min_c]
    conncomp_c = conncomp[:min_r, :min_c]
    coherence_c = coherence_ml[:min_r, :min_c]

    if conncomp_c[REF_PIXEL] == 0:
        print(f"  {d1} -> {d2}: reference pixel not reliable -- EXCLUDING")
        excluded_pairs.append((d1, d2))
        continue

    bridged, offsets = bridge_unwrap_regions(
        unwrapped_c, conncomp_c, bridge_radius=50, min_region_size=100, reference_pixel=REF_PIXEL,
    )
    sbas_pairs.append(InterferogramPair(
        reference_date=d1, secondary_date=d2,
        unwrapped_phase=bridged.astype(np.float32), coherence=coherence_c.astype(np.float32),
    ))
    print(f"  {d1} -> {d2}: bridged and included")

print(f"\n{len(sbas_pairs)}/{len(network_pairs)} pairs usable; excluded: {excluded_pairs}")
network_check = DataValidator.validate_sbas_network(sbas_pairs, extracted_dates)
print(f"Network valid: {network_check.valid}")

## 12. SBAS velocity + LOS-to-vertical conversion

In [ ]:
if network_check.valid and len(sbas_pairs) >= 2:
    sbas = SBASTimeSeries(reference_date=sorted(sbas_pairs, key=lambda p: p.reference_date)[0].reference_date, use_gpu=False)
    ts_result = sbas.invert(sbas_pairs, coherence_threshold=0.3, reference_pixel=REF_PIXEL)

    vertical_cm_yr = los_to_vertical_displacement(ts_result.velocity, incidence_angle_deg=INCIDENCE_ANGLE_DEG) * 100
    print(f"Real LOS velocity range: [{np.nanmin(ts_result.velocity)*1000:.1f}, {np.nanmax(ts_result.velocity)*1000:.1f}] mm/year")
    print(f"Vertical-equivalent velocity range: [{np.nanmin(vertical_cm_yr):.1f}, {np.nanmax(vertical_cm_yr):.1f}] cm/year")
    print(f"\nFor comparison, Cigna & Tapete (2021): peak -39.1 cm/year in Iztapalapa (300+ scenes, 2014-2020)")
    result_map = vertical_cm_yr
else:
    print("Network not connected enough for real SBAS -- reporting the single best pair's LOS displacement instead.")
    best_pair = max(sbas_pairs, key=lambda p: reliability.get((p.reference_date, p.secondary_date), 0)) if sbas_pairs else None
    result_map = (best_pair.unwrapped_phase * WAVELENGTH_M / (4 * np.pi) * 100) if best_pair else None

## 13. Real map of the result

In [ ]:
if result_map is not None:
    disp_path = output_dir / "velocity_cm_yr.tif"
    profile = {
        "driver": "GTiff", "count": 1, "height": result_map.shape[0], "width": result_map.shape[1],
        "crs": interferograms[reference_pair].profile.get("crs"), "transform": reference_transform,
        "dtype": "float32", "nodata": -9999.0,
    }
    with rasterio.open(disp_path, "w", **profile) as dst:
        dst.write(result_map.astype(np.float32)[np.newaxis])

    with rasterio.open(disp_path) as src:
        bounds = src.bounds
    center_lat, center_lon = (bounds.bottom + bounds.top) / 2, (bounds.left + bounds.right) / 2

    mv = MapViewer(center=(center_lat, center_lon), zoom=12)
    mv.add_basemap("SATELLITE")
    vmin, vmax = float(np.nanpercentile(result_map, 2)), float(np.nanpercentile(result_map, 98))
    mv.add_raster(str(disp_path), colormap="RdBu_r", layer_name="vertical_velocity_cm_yr", vmin=vmin, vmax=vmax)
    mv.show()

## 14. Honest summary

**Published reference**: Cigna & Tapete (2021), Remote Sensing of
Environment — peak -39.1 cm/year in Iztapalapa, from 300+ Sentinel-1
scenes, 2014-2020, a real, full 6-year SBAS network.

This notebook searches, downloads, and processes real, fresh data end to
end -- not a replication, a real, honest test of whether the sign and
rough magnitude of subsidence here are directionally consistent with the
published result, using far less data.